In [ ]:
!pip install -q koreanize-matplotlib

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 설정: 본인 드라이브 위치에 맞게 수정 =====
BASE     = '/content/drive/MyDrive/bigcontest2026'
CARD1    = f'{BASE}/신한카드_빅콘테스트2026_데이터1.txt'
CARD2    = f'{BASE}/신한카드_빅콘테스트2026_데이터2_수정.txt'
FLOW_DIR = f'{BASE}/flow'                       # 유동인구 CSV 18개가 있는 폴더
WEATHER  = f'{BASE}/weather_daily_2025H2.csv'

In [ ]:
import os

# 파일 있는지 확인 (True면 OK)
for p in [CARD1, CARD2, WEATHER]:
    print(os.path.exists(p), p)

# flow 폴더 안 파일 개수 (18개면 OK)
print(len(os.listdir(FLOW_DIR)), '개')

In [ ]:
# 파일 이름 순서대로 출력
for f in sorted(os.listdir(FLOW_DIR)):
    print(f)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프 한글 깨짐 방지

# 카드 데이터 (한글 인코딩 cp949, 탭으로 구분)
d1 = pd.read_csv(CARD1, sep='\t', encoding='cp949')
d2 = pd.read_csv(CARD2, sep='\t', encoding='cp949')

print(d1.shape, d2.shape)  # (행 수, 열 수)
d1.head()

In [ ]:
# 열마다 값 종류 확인
for c in ['TIME_GB', 'MCT_SGG_CD', 'SEX_CCD', 'AGE_CCD']:
    print(c, d1[c].unique())

print('업종 수:', d1['MCT_RY_CD'].nunique())
print('날짜:', d1['TA_YMD'].min(), '~', d1['TA_YMD'].max(), '/', d1['TA_YMD'].nunique(), '일')

In [ ]:
# 각 값이 몇 행인지 (많은 순)
for c in ['TIME_GB', 'MCT_SGG_CD', 'SEX_CCD', 'AGE_CCD']:
    print(d1[c].value_counts(), '\n')

In [ ]:
# 가능한 조합 수 = 날짜 × 시간대 × 업종 × (성별2 × 연령9 + 법인1)
days = d1['TA_YMD'].nunique()
inds = d1['MCT_RY_CD'].nunique()
max_cells = days * 4 * inds * (2 * 9 + 1)

for r in ['서울 강남구', '강원 춘천시']:
    n = (d1['MCT_SGG_CD'] == r).sum()
    print(r, n, '/', max_cells, '→ 채워진 비율', round(n / max_cells * 100, 1), '%')

In [ ]:
# 업종 하나당 가능한 칸 수
max_per_ind = days * 4 * (2 * 9 + 1)

fill = d1.groupby(['MCT_RY_CD', 'MCT_SGG_CD']).size().unstack() / max_per_ind * 100
fill = fill.round(1).sort_values('강원 춘천시')

print(fill.head(20))  # 가장 비어 있는 업종
print(fill.tail(10))  # 가장 꽉 찬 업종

In [ ]:
d1['date'] = pd.to_datetime(d1['TA_YMD'].astype(str))  # 날짜 형식으로
d1p = d1[d1['SEX_CCD'] != '법인']  # 개인 결제만 (법인 빼기)
print(len(d1), len(d1p))

In [ ]:
gn = d1p[d1p['MCT_SGG_CD'] == '서울 강남구'].copy()
gn['주말'] = gn['date'].dt.dayofweek >= 5  # 토(5), 일(6)

# 새벽(00~05시) 결제금액 비중 %
dawn = gn[gn['TIME_GB'] == '00_05'].groupby('MCT_RY_CD')['TS_AT'].sum() / gn.groupby('MCT_RY_CD')['TS_AT'].sum() * 100

# 주말 하루 평균 / 평일 하루 평균
day = gn.groupby(['MCT_RY_CD', 'date', '주말'])['TS_AT'].sum().reset_index()
wk = day.groupby(['MCT_RY_CD', '주말'])['TS_AT'].mean().unstack()
weekend_ratio = wk[True] / wk[False]

check = pd.DataFrame({'새벽%': dawn.round(1), '주말/평일': weekend_ratio.round(2)})
print(check.sort_values('새벽%', ascending=False).head(15))
print(check.sort_values('주말/평일').head(15))

In [ ]:
pd.set_option('display.max_rows', 100)  # 최대 100행까지 다 보이게
print(check.sort_values('주말/평일'))

In [ ]:
import glob

# 요일 파일 6개 합치기
files = sorted(glob.glob(f'{FLOW_DIR}/flow_wkdy_pop_*.csv'))
fw = pd.concat([pd.read_csv(f, sep='|', encoding='utf-8-sig', dtype={'BLOCK_CD': str, 'STD_YM': str}) for f in files], ignore_index=True)

print(len(files), '개 파일', fw.shape)
print(fw.head())

In [ ]:
# 소지역코드 앞 5자리 = 시군구 코드 (11230 강남, 32010 춘천)
fw['region'] = fw['BLOCK_CD'].str[:5].map({'11230': '강남', '32010': '춘천'})

print(fw['region'].value_counts(dropna=False))  # NaN = 강남·춘천 아닌 격자

fw = fw[fw['region'].notna()]  # 강남·춘천만 남기기
print(fw.shape)